# Breast Cancer - Scaling And Classifier Comparison

Notebook này tập trung vào 1 dataset, chạy nhiều mô hình để so sánh và ghi lại kết quả thực nghiệm.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.svm import SVC

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "dataset"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")


In [ ]:
df = pd.read_csv(DATA_DIR / "breast_cancer.csv")
display(df.head())

X = df.drop(columns=["target"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [ ]:
models = {
    "LogisticRegression": Pipeline([("scaler", RobustScaler()), ("model", LogisticRegression(max_iter=2000))]),
    "SVC": Pipeline([("scaler", StandardScaler()), ("model", SVC())]),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, preds),
        "f1_weighted": f1_score(y_test, preds, average="weighted"),
    })

results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
display(results_df)
sns.barplot(data=results_df, x="accuracy", y="model", palette="Greens_r")
plt.title("Breast Cancer Model Comparison")
plt.show()
